In [1]:
import pandas as pd


In [2]:
df = pd.DataFrame({'text': ['lottery', 'money', 'hyderabad', 'job opening'],
                   'type': ['spam', 'spam', 'ham', 'ham']
                   })

In [3]:
df

,text,type
0,lottery,spam
1,money,spam
2,hyderabad,ham
3,job opening,ham


Simple RNN for understanding

In [4]:
from sklearn.preprocessing import LabelEncoder # LabelEncoder will take a string and convert it to numerical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

Preprocess data

In [5]:
label_encoder = LabelEncoder()
df['type'] = label_encoder.fit_transform(df['type']) # Converting categorical values to numerical with LabelEncoder rather than using replace

In [6]:
df

,text,type
0,lottery,1
1,money,1
2,hyderabad,0
3,job opening,0


In [7]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['text'])

In [8]:
X = tokenizer.texts_to_sequences(df['text']) # Because we are working on RNN and that only works on sequence data. If we don't use sequence then it will put random data in corpus

For text data we use 'post' padding whereas for image data we use 'pre'.

For text if we ues 'pre' then it will add 0 at the start and that will not be efficient.

In [9]:
X = pad_sequences(X, padding='post')

In [10]:
X # See 6:41 AM for description

array([[1, 0],
       [2, 0],
       [3, 0],
       [4, 5]], dtype=int32)

In [11]:
y = df['type'].values
y


array([1, 1, 0, 0])

Build RNN Architecture

Vocabulary size is 5 words plus one padding

In [12]:
vocab_size = len(tokenizer.word_index) + 1
vocab_size

6

Embedding dimensions
4 rows and 2 columns which makes it 8 dimensions.
Embedding is a Matrix

[[1, 0],
  [2, 0],
  [3, 0],
  [4, 5]]

In [13]:
embedding_dim = 8


In [14]:
X.shape

(4, 2)

In [15]:
input_length = X.shape[1] # Length of each sequence
input_length

2

BUILD THE MODEL

In [122]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [20]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=input_length)) # input_dim=vocab_size is the number of neurons in this layer (here vocab_size is 6 so 6 neurons)
model.add(SimpleRNN(2, activation='tanh'))
model.add(Dense(1, activation='sigmoid'))

In [21]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [28]:
model.fit(X, y, epochs=5, verbose=1)

Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 1.0000 - loss: 0.6350
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 1.0000 - loss: 0.6317
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 1.0000 - loss: 0.6285
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 1.0000 - loss: 0.6252
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 1.0000 - loss: 0.6218


In [32]:
new_input = tokenizer.texts_to_sequences(['lottery'])
new_input = pad_sequences(new_input, padding='post', maxlen=input_length) # need to send similar size/dimension for input length

In [30]:
prediction = model.predict(new_input)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step


array([[0.533332]], dtype=float32)

In [31]:
print('prediction for lottery ', prediction)

prediction for lottery  [[0.533332]]


Second Example: With Sample data (SPAM DATA)

https://www.kaggle.com/datasets/tmehul/spamcsv?resource=download

In [33]:
import pandas as pd


In [74]:
#df = pd.read_csv('/content/spam.csv', encoding='latin-1') # First option or
df = pd.read_csv('/content/spam.csv', encoding_errors='ignore')

In [75]:
df

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,NaN,NaN,NaN
5568,ham,Will _ b going to esplanade fr home?,NaN,NaN,NaN
5569,ham,"Pity, * was in mood for that. So...any other s...",NaN,NaN,NaN
5570,ham,The guy did some bitching but I acted like i'd...,NaN,NaN,NaN


In [76]:
df.columns

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')

In [77]:
df = df[['v2', 'v1']] # select first two columns

In [78]:
df.head()

,v2,v1
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham


In [79]:
# change the column name

df = df.rename(columns={'v2': 'text', 'v1': 'type'})

In [80]:
df.head()

,text,type
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam
3,U dun say so early hor... U c already then say...,ham
4,"Nah I don't think he goes to usf, he lives aro...",ham


In [81]:
df.size

11144

In [82]:
tokenizer = Tokenizer(num_words = 5000, oov_token='<OOV>') # OUT OF VOCABULARY, When a word is not in vocabulary a label is assigned <OOV>
tokenizer.fit_on_texts(df['text'])
#

In [83]:
sequences = tokenizer.texts_to_sequences(df['text'])

In [84]:
max_sequence_length = 100
X = pad_sequences(sequences, maxlen=max_sequence_length, padding='post')

In [86]:
df['type'].values

array(['ham', 'ham', 'spam', ..., 'ham', 'ham', 'ham'], dtype=object)

In [87]:
y = df['type'].values
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
y

array([0, 0, 1, ..., 0, 0, 0])

In [88]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [123]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=max_sequence_length))
#model.add(SimpleRNN(256, activation='tanh'))
model.add(LSTM(256, activation='tanh'))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [124]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [125]:
history = model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.8688 - loss: 0.4265 - val_accuracy: 0.8655 - val_loss: 0.3961
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8754 - loss: 0.3774 - val_accuracy: 0.8655 - val_loss: 0.3961
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8704 - loss: 0.3871 - val_accuracy: 0.8655 - val_loss: 0.3963
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8614 - loss: 0.4029 - val_accuracy: 0.8655 - val_loss: 0.3993
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8661 - loss: 0.3982 - val_accuracy: 0.8655 - val_loss: 0.3958


In [126]:
sample_message = """Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"""



In [127]:
# Conver Sample into sequences
sample_sequence = tokenizer.texts_to_sequences([sample_message])
sample_sequence = pad_sequences(sample_sequence, maxlen=max_sequence_length, padding='post')

In [128]:
prediction = model.predict(sample_sequence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step


In [129]:
prediction

array([[0.1283455]], dtype=float32)

In [130]:
a = int(prediction[0][0])

In [131]:
if a == 1:
  print('It is a SPAM')
else:
  print('It is not a SPAM')

It is not a SPAM
